# Coding attention mechanisms

Sequence processing one item at a time fails due to lack of context (think: translating a text string). Problem is commonly addressed by using encoder-decoder architectures. 

Problem with RNNs is that they process text sequentially, trying to capture the entire meaning of the input sequence from the final hidden state. Thus, a big limitation of RNNs is that they are unable to directly access earlier hidden states from the encoder during the decoding phase. Consequently, it relies solely on the current hidden state to encapsulate all relevant information. The RNN must remember the entire encoded input input in a single hidden state before passing it to the decoder. This can lead to loss of context, especially in sequences with dependencies spanning long distances.

Alternatively, the self-attention mechanism of the transformer architecture allows each position in the input sequence to consider the relevancy of all other positions in the same sequence. The self in self-attention refers to the computation of attention weights by relating different positions within a single input sequence. It assesses and learns the relationships and dependencies between various parts of the input itself. 

Self-attention calculates context vectors $z^{(i)}$ for each element $x^{(i)}$ of the input sequence. A context vector is an enriched embedding vector, incorporating information from all other other elements to create an enriched representation of each element in an input sequence. 

In [1]:
import torch
# Sample input sequence already embedded as 3-dimensional vectors 
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your    x^1 
     [0.55, 0.87, 0.66], # journey x^2
     [0.57, 0.85, 0.64], # starts  x^3
     [0.22, 0.58, 0.33], # with    x^4
     [0.77, 0.25, 0.10], # one     x^5
     [0.05, 0.80, 0.55]] # step    x^6
)

In [ ]:
# First step to self-attention is to compute intermediate values
# called attention scores. These scores determine how much each token should
# attend to every other token in the sequence.  
query = inputs[1]
# Second token input serves as the query token
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    # Attention scores are computed as the dot product between the 
    # query and each token embedding is the dot product between the 
    # query and each token embedding
    attn_scores_2[i] = torch.dot(query, x_i)

# NOTE: The dot product can be viewed as a measure of similarity as it
# quantifies how closely two vectors are aligned: a high dot product indicates
# a greater degree of alignment. In context of self-attention, dot product
# determines extent to which each element in a sequence focuses on any other
# element: the higher the dot product, the higher the similarity and 
# attention score between two elements.
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [9]:
# Normalize so sum to 1
attn_weights = torch.softmax(attn_scores_2, dim=0)
# Softmax ensures positive output and permits interpretation as 
# probabilities/relative importance
print(attn_weights)
print(attn_weights.sum())

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor(1.)


In [ ]:
# Context vector is computed by multiplying the embedded 
# input tokens x^(i) with the corresponding attention weights z^(2) and
# them summing the resulting vectors. Thus, the context vector, z^(2), is 
# the weighted sum of all input vectors, obtained by multiplying each input
# vector with corresponding attention weight.
# ---
# Second input token is the query
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    # Multiply embedding vector with corresponding attention weight
    context_vec_2 += x_i * attn_weights[i]
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


In [ ]:
# Using matrix multiplication to calculate attention weights
attn_scores = inputs @ inputs.T 
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [12]:
# All context vectors via matrix multiplication
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [13]:
print(context_vec_2)
print(all_context_vecs[1])

tensor([0.4419, 0.6515, 0.5683])
tensor([0.4419, 0.6515, 0.5683])


## Self-attention with trainable weights 

The GPT models use a self-attention variant called scaled dot-product attention. The goal is still to copmute context vectors as weighted sums over input vectors specific to a certain input element. This approach, however, introduces trainable weight matrices $W_q$, $W_k$, $W_v$. Matrices project the input tokens, $x^{(i)}$, into query, key and value vectors. 



In [17]:
# Use second input element for illustration
x_2 = inputs[1]
# Input embedding size, d=3
d_in = inputs.shape[1]
# Output embedding size, d_out=2
d_out = 2

# Initiallize weight matrices
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [ ]:
# Compute query, key and value vectors
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)

tensor([1.0734, 1.8420])


In [20]:
# Obtain keys and values via matrix multiplication
keys = inputs @ W_key 
values = inputs @ W_value 
print(keys.shape)
print(values.shape)

torch.Size([6, 2])
torch.Size([6, 2])


In [22]:
# Compute attention scores
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(3.6305)


In [23]:
# All attention scores for a given query
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([2.9105, 3.6305, 3.5899, 1.9338, 1.8477, 2.4630])


In [25]:
# Attention weights by scaling attention scores and using softmax
# Note that scale scores by dividing by square root of embedding dimension 
# of the keys
d_k = keys.shape[-1]
# Similar to computing context vector as weighted sum over input vectors,
# now compute context vector as weighted sum over value vectors
attn_weights_2 = torch.softmax(attn_scores_2 / d_k ** 0.5, dim=-1)
print(attn_weights_2)

tensor([0.1672, 0.2781, 0.2703, 0.0838, 0.0788, 0.1218])


In [26]:
# In matrix multiplication
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.8053, 0.9018])


## Self-attention class

In [31]:
import torch.nn as nn


class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        # Initialize trainable weight matrices
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key =   nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key 
        queries = x @ self.W_query
        values = x @ self.W_value
        attn_scores = queries @ keys.T 
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.2, dim=-1
        )
        context_vec = attn_weights @ values 
        return context_vec
    
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.3038, 0.8154],
        [0.3115, 0.8336],
        [0.3112, 0.8328],
        [0.2980, 0.8018],
        [0.2956, 0.7961],
        [0.3031, 0.8139]], grad_fn=<MmBackward0>)


In [32]:
# Improving self-attention by utilizing improve weight initialization 
# in nn.Linear layers, and effectively perform matrix multiplication
# when bias units are disabled.


class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        # Initialize trainable weight matrices
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key =   nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys =    self.W_key(x)
        queries = self.W_query(x)
        values =  self.W_value(x)
        attn_scores = queries @ keys.T 
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.2, dim=-1
        )
        context_vec = attn_weights @ values 
        return context_vec
    
torch.manual_seed(123)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.5351, -0.1049],
        [-0.5334, -0.1084],
        [-0.5334, -0.1083],
        [-0.5301, -0.1079],
        [-0.5318, -0.1067],
        [-0.5304, -0.1085]], grad_fn=<MmBackward0>)


## Hiding future words with causal attention